In [4]:
#prep relevent packages 
from pathlib import Path
import pandas as pd
import re

In [5]:
# package to add to kernel to read pdfs. 
# Can also do %conda install pdfplumber if running on windows 
try:
    import pdfplumber
except Importerror:
    %pip install pdfplumber
    import pdfplumber 

In [6]:
# file path to import data
root_fldr = Path.cwd().parent
raw_fldr = root_fldr / 'data' / 'raw' 
processed_fldr = root_fldr / 'data' / 'processed'

# import cfp residential risk pdf 
residential_risk = raw_fldr / 'cfp_data' / 'cfp_5_year_residential_risk.pdf'

In [ ]:
table_settings = {
    'vertical_strategy': 'text',
    'horizontal_strategy': 'text',
}
rows = []  # reset each run so reruns don't duplicate rows

#help provided with some AI, particularly with zip_cell check to get unified, scraped dataframe
with pdfplumber.open(residential_risk) as pdf:
    for page in pdf.pages:
        for table in page.extract_tables(table_settings):
            for row in table:
                zip_cell = (row[0] or '').strip()
                if re.fullmatch(r'\d{5}', zip_cell):
                    # clean each cell: None -> '', drop spaces, commas, and %
                    clean_row = [(cell or '').replace(' ', '').replace(',', '').replace('%', '')
                                 for cell in row]
                    rows.append(clean_row)

#need to add 4 $ columns to align with the scraping of the pdf 
cols = ['zip',
        'growth_2025', '$', 'exposure_2025',
        'growth_2024', '$', 'exposure_2024',
        'growth_2023', '$', 'exposure_2023',
        'growth_2022', '$', 'exposure_2022',
        'growth_2021', 'exposure_2021']
residential_risk = pd.DataFrame(rows, columns=cols)


In [9]:
#check dataframe to see if it can be read  
residential_risk.head()

,zip,growth_2025,$,exposure_2025,growth_2024,$,exposure_2024,growth_2023,$,exposure_2023,growth_2022,$,exposure_2022,growth_2021,exposure_2021
0,94501,146,$,242456273,68,$,98431342,18,$,58719416,14,$,49797731,$,43791971
1,94502,43,$,9826954,1327,$,6880050,2,$,481983,185,$,470279,$,165000
2,94536,108,$,84445812,199,$,40642190,85,$,13571624,31,$,7327808,$,5605823
3,94538,85,$,28863175,144,$,15574256,165,$,6370385,77,$,2406677,$,1358996
4,94539,114,$,171025425,193,$,79814473,98,$,27207162,270,$,13722261,$,3712311


In [ ]:
#getting rid of $ columns 
residential_risk = residential_risk.loc[:, residential_risk.columns != '$'] #checking all columns and mainting all those that aren't '$'
residential_risk.head()

,zip,growth_2025,exposure_2025,growth_2024,exposure_2024,growth_2023,exposure_2023,growth_2022,exposure_2022,growth_2021,exposure_2021
0,94501,146,242456273,68,98431342,18,58719416,14,49797731,$,43791971
1,94502,43,9826954,1327,6880050,2,481983,185,470279,$,165000
2,94536,108,84445812,199,40642190,85,13571624,31,7327808,$,5605823
3,94538,85,28863175,144,15574256,165,6370385,77,2406677,$,1358996
4,94539,114,171025425,193,79814473,98,27207162,270,13722261,$,3712311


In [16]:
#converting df into appropriate variable types 

residential_risk_clean = (
    residential_risk
    .apply(pd.to_numeric, errors='coerce') #converting strings to numbers, '-' becomes NaN
    .assign( #converting into percentages 
        growth_2025 = lambda d: d['growth_2025'] / 100,
        growth_2024 = lambda d: d['growth_2024'] / 100,
        growth_2023 = lambda d: d['growth_2023'] / 100,
        growth_2022 = lambda d: d['growth_2022'] / 100 
    )
    .astype({
        'growth_2025': 'float64',
        'exposure_2025': 'Int64',
        'growth_2024': 'float64',
        'exposure_2024': 'Int64',
        'growth_2023': 'float64',
        'exposure_2023': 'Int64',
        'growth_2022': 'float64',
        'exposure_2022': 'Int64',
        'exposure_2021': 'Int64'
    })
)

residential_risk_clean.head()

,zip,growth_2025,exposure_2025,growth_2024,exposure_2024,growth_2023,exposure_2023,growth_2022,exposure_2022,growth_2021,exposure_2021
0,94501,1.46,242456273,0.68,98431342,0.18,58719416,0.14,49797731,NaN,43791971
1,94502,0.43,9826954,13.27,6880050,0.02,481983,1.85,470279,NaN,165000
2,94536,1.08,84445812,1.99,40642190,0.85,13571624,0.31,7327808,NaN,5605823
3,94538,0.85,28863175,1.44,15574256,1.65,6370385,0.77,2406677,NaN,1358996
4,94539,1.14,171025425,1.93,79814473,0.98,27207162,2.70,13722261,NaN,3712311


In [15]:
residential_risk_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1679 entries, 0 to 1678
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   zip            1679 non-null   int64  
 1   growth_2025    1679 non-null   float64
 2   exposure_2025  1660 non-null   Int64  
 3   growth_2024    1679 non-null   float64
 4   exposure_2024  1650 non-null   Int64  
 5   growth_2023    1679 non-null   float64
 6   exposure_2023  1599 non-null   Int64  
 7   growth_2022    1679 non-null   float64
 8   exposure_2022  1569 non-null   Int64  
 9   growth_2021    0 non-null      float64
 10  exposure_2021  1543 non-null   Int64  
dtypes: Int64(5), float64(5), int64(1)
memory usage: 152.6 KB


In [ ]:
#check to see if numbers from pull are aligned with pdf 
# Plans from pdf
# 2025 exposure = $645,115,692,650
# 2024 exposure = $431,066,598,171
# 2023 exposure = $270,510,184,565
total_2025_exposure = residential_risk_clean['exposure_2025'].sum()
total_2024_exposure = residential_risk_clean['exposure_2024'].sum()
total_2023_exposure = residential_risk_clean['exposure_2023'].sum()

print('2025 exposure:',total_2025_exposure)
print('2024 exposure:',total_2024_exposure)
print('2023 exposure:',total_2023_exposure)



2025 exposure: 645115692650
2024 exposure: 431066598171
2023 exposure: 270510184565


In [18]:
#converting to processed folder csv 
residential_risk_clean.to_csv(processed_fldr / 'cfp_residential_risk_cleaned.csv')
